# Validação de Estimativa Volumétrica e Transferência entre Talhões

Este notebook consolida e plota os resultados dos 4 experimentos de validação do pipeline biométrico em nuvens LiDAR da Fazenda Canoas (Talhões 37, 62 e 63).

## Objetivos da Avaliação
1. **Validação cruzada sem vazamento de dados (Leave-One-Plot-Out) e Saturação Amostral:** Avaliar o erro de generalização em parcelas não vistas durante a calibração no Talhão 37 ($10\text{ cm}$) e medir como o erro varia em função do número de parcelas de treino ($N \in [2, 4, 6, 8, 10]$).
2. **Transferência Intra-Clone na Mesma Resolução ($5\text{ cm}$):** Calibrar os hiperparâmetros volumétricos nas 4 parcelas do Talhão 62 ($25.96\text{ ha}$) e avaliar a extrapolação na área amostral do Talhão 63 ($15.04\text{ ha}$, mesmo clone `AEC0144` e mesmo espaçamento $3,50 \times 2,50\text{ m}$).
3. **Sensibilidade à Resolução da Nuvem ($5\text{ cm}$ vs $10\text{ cm}$):** Comparar as estimativas volumétricas nas mesmas parcelas físicas do Talhão 62 utilizando os voos de $5\text{ cm}$ e $10\text{ cm}$.
4. **Transferência Inter-Talhões na Mesma Resolução ($10\text{ cm}$):** Investigar se o viés observado anteriormente na transferência entre o Talhão 37 e o Talhão 62 decorria da diferença de esparcidade das nuvens ($10\text{ cm} \leftrightarrow 5\text{ cm}$), avaliando a transferência direta com ambos em $10\text{ cm}$.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.titlesize": 14,
    "figure.titleweight": "bold",
})

RES_DIR = os.path.abspath("../results")
if not os.path.exists(os.path.join(RES_DIR, "exp1_t37")):
    RES_DIR = "/mnt/scratch/dataset/euc/canoa/results/volume_validation"
FIGS_DIR = os.path.join(RES_DIR, "figures")
os.makedirs(FIGS_DIR, exist_ok=True)
print("Results directory:", RES_DIR)

---
## Experimento 1: Validação Cruzada (LOPO) e Curva de Saturação no Talhão 37 ($10\text{ cm}$)

### O que almejamos
Eliminar o uso das mesmas parcelas na calibração e no teste (data leakage) através de validação Leave-One-Plot-Out (11 folds, 118 árvores pareadas) e verificar quantas parcelas de treino são necessárias até a estabilização do erro nas parcelas de teste (20 repetições Monte Carlo por tamanho amostral).

### Resultados
- **Leave-One-Plot-Out (11 parcelas):** Erro percentual absoluto médio (MAE) de **34.38%** no modo Global Best (`voxel_vs0.08`) e **22.40%** no modo Human-in-the-Loop.
- **Curva de Saturação:** O erro médio nas parcelas de teste passa de 32.4% ($N=2$) e 34.0% ($N=4$) para 31.4% ($N=10$).

### Implicações
- Como o modelo global possui baixa complexidade paramétrica (escolha do tamanho de voxel ou fator de forma ótimo), aumentar o conjunto de calibração além de 4 parcelas traz redução marginal no erro médio de seleção do modelo global.

In [ ]:
df_sat = pd.read_csv(os.path.join(RES_DIR, "exp1_t37/sampling_saturation_summary.csv"))
df_lopo = pd.read_csv(os.path.join(RES_DIR, "exp1_t37/lopo_results_t37.csv"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Subplot 1: Saturation curve
ax1.plot(df_sat["n_train_plots"], df_sat["global_err_mean"], marker="o", color="#1b4d3e", linewidth=2.2, markersize=7, label="Global Mode (Held-Out MAE %)")
ax1.plot(df_sat["n_train_plots"], df_sat["human_err_mean"], marker="s", linestyle="--", color="#16a085", linewidth=2.0, markersize=7, label="Human-in-the-Loop (Held-Out MAE %)")
ax1.fill_between(
    df_sat["n_train_plots"],
    df_sat["global_err_mean"] - df_sat["global_err_std"],
    df_sat["global_err_mean"] + df_sat["global_err_std"],
    color="#1b4d3e",
    alpha=0.12,
    label="±1 Std Dev (20 Monte Carlo splits)"
)
ax1.set_xlabel("Number of Calibration Plots in Training Set (N_train)")
ax1.set_ylabel("Held-Out Plot Volumetric Error (Mean Absolute %)")
ax1.set_title("Sampling Saturation Curve (Talhão 37 — 10 cm)")
ax1.set_xticks(df_sat["n_train_plots"])
ax1.legend(loc="upper right")
ax1.grid(True, linestyle="--", alpha=0.5)

# Subplot 2: LOPO Error across the 11 folds
x_plots = np.arange(len(df_lopo))
w = 0.38
ax2.bar(x_plots - w/2, df_lopo["err_global_ha_pct"].abs(), width=w, color="#34495e", label=f"Global Best (Mean: {df_lopo['err_global_ha_pct'].abs().mean():.1f}%)")
ax2.bar(x_plots + w/2, df_lopo["err_human_ha_pct"].abs(), width=w, color="#16a085", label=f"Human-in-the-Loop (Mean: {df_lopo['err_human_ha_pct'].abs().mean():.1f}%)")
ax2.set_xticks(x_plots)
ax2.set_xticklabels(df_lopo["test_plot_id"].astype(int), rotation=45)
ax2.set_xlabel("Held-Out Test Plot ID")
ax2.set_ylabel("Absolute Volumetric Error (%)")
ax2.set_title("Leave-One-Plot-Out (11-Fold) Test Error per Plot")
ax2.legend(loc="upper right")
ax2.grid(True, linestyle="--", alpha=0.5, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, "figure1_sampling_saturation.png"), dpi=300)
plt.show()

display(df_sat)
display(df_lopo[["test_plot_id", "n_test_trees", "calibrated_config", "gt_m3_ha", "pred_global_m3_ha", "err_global_ha_pct", "pred_human_m3_ha", "err_human_ha_pct"]])

---
## Experimento 2: Transferência Intra-Clone na Mesma Resolução (Talhão 62 $\rightarrow$ Talhão 63 em $5\text{ cm}$)

### O que almejamos
Separar os dados cadastrais dos Talhões 62 ($25.96\text{ ha}$, 4 parcelas de campo) e 63 ($15.04\text{ ha}$, sem parcelas de campo), calibrar o estimador volumétrico nas árvores pareadas do Talhão 62 e aplicá-lo à nuvem amostral do Talhão 63 sob a mesma densidade de pontos ($5\text{ cm}$).

### Resultados
- **Calibração no T62 (40 árvores):** Melhor configuração global `voxel_vs0.07` ($\text{RMSE} = 0.1709\text{ m}^3$).
- **Extrapolação no T63 (33 árvores QC):** Volume médio estimado de $0.3094\text{ m}^3$/árvore, resultando em **$201.14\text{ m}^3/\text{ha}$** ($3.025,2\text{ m}^3$ em $15.04\text{ ha}$), frente à referência de inventário de **$201.29\text{ m}^3/\text{ha}$** ($3.027,4\text{ m}^3$ total), uma diferença relativa de **-0.07%**.

### Implicações
- Quando a resolução do voo ($5\text{ cm}$), o material genético (`AEC0144`) e o regime de plantio são mantidos constantes, o tamanho de voxel calibrado no T62 preserva consistência ao ser aplicado na amostra do T63. Ressalta-se que o valor de referência oficial do inventário ($201.29\text{ m}^3/\text{ha}$) é reportado de forma agregada para o bloco 062-063.

In [ ]:
with open(os.path.join(RES_DIR, "exp2_t62_t63/exp2_summary.json")) as f:
    exp2_meta = json.load(f)

t62_global = exp2_meta["extrapolation_results"][0]
t63_entry = exp2_meta["extrapolation_results"][2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

labels = ["Field Inventory Reference", "Transferred Model (voxel_vs0.07)"]
yield_vals = [t63_entry["gt_m3_ha"], t63_entry["pred_m3_ha"]]
bars1 = ax1.bar(labels, yield_vals, color=["#2c3e50", "#2980b9"], width=0.45, edgecolor="black", linewidth=1.0)
for b in bars1:
    y = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2.0, y + 3, f"{y:.1f} m³/ha", ha="center", va="bottom", fontweight="bold", fontsize=11)

ax1.set_ylim(0, 240)
ax1.set_ylabel("Volumetric Yield (m³/ha)")
ax1.set_title(f"Talhão 63 Volumetric Density (Rel. Diff: {t63_entry['err_pct']:+.1f}%)")
ax1.grid(True, linestyle="--", alpha=0.5, axis="y")

stand_labels = ["Talhão 62 (25.96 ha)\n[Calibration Stand]", "Talhão 63 (15.04 ha)\n[Unvisited Target Stand]"]
gt_totals = [t62_global["gt_total_m3"], t63_entry["gt_total_m3"]]
pred_totals = [t62_global["pred_total_m3"], t63_entry["pred_total_m3"]]

x_s = np.arange(2)
w_s = 0.35
b_gt = ax2.bar(x_s - w_s/2, gt_totals, width=w_s, color="#2c3e50", label="Field Inventory Reference")
b_pr = ax2.bar(x_s + w_s/2, pred_totals, width=w_s, color="#2980b9", label="LiDAR Global Model (voxel_vs0.07)")
for b in list(b_gt) + list(b_pr):
    y = b.get_height()
    ax2.text(b.get_x() + b.get_width()/2.0, y + 80, f"{y:,.0f} m³", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax2.set_xticks(x_s)
ax2.set_xticklabels(stand_labels)
ax2.set_ylim(0, 6200)
ax2.set_ylabel("Total Extrapolated Volume (m³)")
ax2.set_title("Total Volume Extrapolation (T62 & T63 — 5 cm)")
ax2.legend(loc="upper right")
ax2.grid(True, linestyle="--", alpha=0.5, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, "figure2_stand_transferability.png"), dpi=300)
plt.show()

df_exp2 = pd.read_csv(os.path.join(RES_DIR, "exp2_t62_t63/extrapolation_summary_t62_t63.csv"))
display(df_exp2)

---
## Experimento 3: Sensibilidade à Resolução da Nuvem ($5\text{ cm}$ vs $10\text{ cm}$ no Talhão 62)

### O que almejamos
Avaliar diretamente o efeito da esparcidade da nuvem de pontos sobre os métodos volumétricos nas mesmas 4 parcelas do Talhão 62 (500, 503, 508, 509).

### Resultados
- **Nuvem de $5\text{ cm}$ (~400 pts/m², 40 árvores):** Melhor configuração `voxel_vs0.07`, $\text{RMSE} = 0.1709\text{ m}^3$, erro médio por parcela de **28.8%** (Global) e **13.8%** (Human-in-the-Loop).
- **Nuvem de $10\text{ cm}$ (~150 pts/m², 38 árvores):** Melhor configuração `voxel_vs0.08`, $\text{RMSE} = 0.1591\text{ m}^3$, erro médio por parcela de **31.4%** (Global) e **27.7%** (Human-in-the-Loop).

### Implicações
- A redução da densidade de pontos de $5\text{ cm}$ para $10\text{ cm}$ desloca o tamanho ótimo de voxel de $7\text{ cm}$ para $8\text{ cm}$ para compensar o menor preenchimento espacial do tronco.
- No modo global, a diferença de erro médio entre as duas resoluções é pequena (28.8% vs 31.4%), mas no modo Human-in-the-Loop a nuvem mais densa ($5\text{ cm}$) permite ajustes melhores árvore a árvore (13.8% vs 27.7%).

In [ ]:
df_trees_res = pd.read_csv(os.path.join(RES_DIR, "exp3_resolution_5vs10cm/trees_resolution_comparison.csv"))
df_plots_res = pd.read_csv(os.path.join(RES_DIR, "exp3_resolution_5vs10cm/plots_resolution_comparison.csv"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

res_types = ["5cm", "10cm"]
global_errs = [df_plots_res[df_plots_res["resolution"] == r]["err_global_ha_pct"].abs().mean() for r in res_types]
human_errs = [df_plots_res[df_plots_res["resolution"] == r]["err_human_ha_pct"].abs().mean() for r in res_types]

x = np.arange(2)
width = 0.35
b1 = ax1.bar(x - width/2, global_errs, width, label="Global Best Config", color="#34495e", edgecolor="black")
b2 = ax1.bar(x + width/2, human_errs, width, label="Human-in-the-Loop", color="#16a085", edgecolor="black")
for b in list(b1) + list(b2):
    y = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2, y + 0.5, f"{y:.1f}%", ha="center", va="bottom", fontweight="bold")

ax1.set_xticks(x)
ax1.set_xticklabels(["5 cm (~400 pts/m²)", "10 cm (~150 pts/m²)"])
ax1.set_ylabel("Mean Absolute Plot Error (%)")
ax1.set_title("Plot-Level Volumetric Error by Point Cloud Resolution (T62)")
ax1.set_ylim(0, 38)
ax1.legend(loc="upper left")
ax1.grid(True, linestyle="--", alpha=0.5, axis="y")

voxel_cols = ["voxel_vs0.06", "voxel_vs0.07", "voxel_vs0.08", "voxel_vs0.09", "voxel_vs0.10"]
voxel_sizes_cm = [6, 7, 8, 9, 10]
sub_5 = df_trees_res[df_trees_res["resolution"] == "5cm"]
sub_10 = df_trees_res[df_trees_res["resolution"] == "10cm"]
rmse_5cm = [np.sqrt(np.mean((sub_5[c] - sub_5["gt_volume"]) ** 2)) for c in voxel_cols]
rmse_10cm = [np.sqrt(np.mean((sub_10[c] - sub_10["gt_volume"]) ** 2)) for c in voxel_cols]

ax2.plot(voxel_sizes_cm, rmse_5cm, marker="o", linewidth=2.2, color="#2980b9", label="5 cm Cloud (Optimal: 7 cm)")
ax2.plot(voxel_sizes_cm, rmse_10cm, marker="s", linewidth=2.2, color="#d35400", label="10 cm Cloud (Optimal: 8 cm)")
ax2.scatter([7], [min(rmse_5cm)], color="#2980b9", s=110, zorder=5)
ax2.scatter([8], [min(rmse_10cm)], color="#d35400", s=110, zorder=5)
ax2.set_xticks(voxel_sizes_cm)
ax2.set_xticklabels([f"{v} cm" for v in voxel_sizes_cm])
ax2.set_xlabel("Voxel Size Parameter")
ax2.set_ylabel("Individual Tree Volume RMSE (m³)")
ax2.set_title("Voxel Size Sensitivity: 5 cm vs 10 cm Point Clouds (T62)")
ax2.legend(loc="upper left")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, "figure3_resolution_comparison.png"), dpi=300)
plt.show()

df_res_sum = pd.read_csv(os.path.join(RES_DIR, "exp3_resolution_5vs10cm/resolution_comparison_summary.csv"))
display(df_res_sum)

---
## Experimento 4: Transferência entre Talhões com Mesma Resolução ($10\text{ cm} \rightarrow 10\text{ cm}$)

### O que almejamos
Verificar a hipótese levantada nas discussões técnicas: aplicar um modelo calibrado em nuvem de $10\text{ cm}$ (Talhão 37) sobre uma nuvem de $5\text{ cm}$ (Talhão 62) introduz viés sistemático devido à diferença de densidade de pontos. Para testar isso, avaliamos a transferência utilizando as nuvens de $10\text{ cm}$ em ambos os talhões.

### Resultados
- **Convergência de Parâmetro:** Tanto o Talhão 37 ($10\text{ cm}$) quanto o Talhão 62 ($10\text{ cm}$) selecionaram independentemente **`voxel_vs0.08`** como o estimador de menor RMSE.
- **Transferência T37 ($10\text{ cm}$) $\rightarrow$ T62 ($5\text{ cm}$) [Resoluções Diferentes]:** Superestimação de **+46.3%** ($294.6\text{ m}^3/\text{ha}$ vs $201.3\text{ m}^3/\text{ha}$), pois o voxel de $8\text{ cm}$ calibrado na nuvem esparsa gera volume excessivo na nuvem densa.
- **Transferência T37 ($10\text{ cm}$) $\rightarrow$ T62 ($10\text{ cm}$) [Mesma Resolução]:** Estimativa de **$200.2\text{ m}^3/\text{ha}$** frente a **$201.3\text{ m}^3/\text{ha}$**, reduzindo o erro relativo de extrapolação para **-0.6%**.
- **Transferência Inversa T62 ($10\text{ cm}$) $\rightarrow$ T37 ($10\text{ cm}$):** O erro passa de -49.4% (quando calibrado em $5\text{ cm}$) para **-25.9%** (quando calibrado em $10\text{ cm}$).

### Implicações
- Confirma-se que a transferência de hiperparâmetros de voxelização exige padronização da densidade de pontos entre as nuvens de calibração e aplicação. Quando a resolução é equiparada em $10\text{ cm}$, o viés artificial de resolução desaparece.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

exp_labels = ["Mismatched Resolution\nT37 (10cm) ➔ T62 (5cm)", "Matched Resolution\nT37 (10cm) ➔ T62 (10cm)"]
error_vals = [46.3, -0.6]
bars_err = ax1.bar(exp_labels, error_vals, color=["#c0392b", "#27ae60"], width=0.45, edgecolor="black", linewidth=1.0)
ax1.axhline(0, color="black", linewidth=1)
for b in bars_err:
    y = b.get_height()
    va = "bottom" if y >= 0 else "top"
    ax1.text(b.get_x() + b.get_width()/2, y + (1.5 if y >= 0 else -3.5), f"{y:+.1f}%", ha="center", va=va, fontweight="bold", fontsize=11)

ax1.set_ylabel("Stand Volumetric Relative Error (%)")
ax1.set_title("Effect of Point Density Matching on Stand Transfer (T37 ➔ T62)")
ax1.set_ylim(-15, 58)
ax1.grid(True, linestyle="--", alpha=0.5, axis="y")

vol_labels = ["Field Inventory\nReference", "Mismatched Transfer\n(10cm ➔ 5cm)", "Matched Transfer\n(10cm ➔ 10cm)"]
yield_vals = [201.3, 294.6, 200.2]
bars_v = ax2.bar(vol_labels, yield_vals, color=["#2c3e50", "#c0392b", "#2980b9"], width=0.45, edgecolor="black", linewidth=1.0)
for b in bars_v:
    y = b.get_height()
    ax2.text(b.get_x() + b.get_width()/2, y + 4, f"{y:.1f} m³/ha", ha="center", va="bottom", fontweight="bold")

ax2.set_ylabel("Volumetric Density (m³/ha)")
ax2.set_title("Talhão 62 Extrapolated Volume (Calibrated on T37)")
ax2.set_ylim(0, 340)
ax2.grid(True, linestyle="--", alpha=0.5, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, "figure4_cross_stand_10cm_transfer.png"), dpi=300)
plt.show()

df_exp4 = pd.read_csv(os.path.join(RES_DIR, "exp4_cross_stand_10cm/cross_stand_resolution_matching_summary.csv"))
display(df_exp4)